In [1]:
import os
from huggingface_hub import InferenceClient

d:\Workspace\AI with Kligs\intelligenza-artificiale\HF Agent course\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
HF_TOKEN = None
with open("./TOKEN", 'r') as f:
    HF_TOKEN = f.read()

In [3]:
client = InferenceClient(model="meta-llama/Llama-4-Scout-17B-16E-Instruct", token=HF_TOKEN)

In [5]:
# This system prompt is a bit more complex and actually contains the function description already appended.
# Here we suppose that the textual description of the tools has already been appended.

SYSTEM_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

get_weather: Get the current weather in a given location

The way you use the tools is by specifying a json blob.
Specifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).

The only values that should be in the "action" field are:
get_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}
example use :

{{
  "action": "get_weather",
  "action_input": {"location": "New York"}
}}


ALWAYS use the following format:

Question: the input question you must answer
Thought: you should always think about one action to take. Only one action at a time in this format:
Action:

$JSON_BLOB (inside markdown cell)

Observation: the result of the action. This Observation is unique, complete, and the source of truth.
(this Thought/Action/Observation can repeat N times, you should take several steps when needed. The $JSON_BLOB must be formatted as markdown and only use a SINGLE action at a time.)

You must always end your output with the following format:

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Now begin! Reminder to ALWAYS use the exact characters `Final Answer:` when you provide a definitive answer. """

In [6]:
output = client.chat.completions.create(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "The capital of France is"},
    ],
    stream=False,
    max_tokens=1024,
)
print(output.choices[0].message.content)

Thought: This is a simple geography question, but I'm not provided with a tool to look up capitals directly. However, I can try to see if there's any indirect way or if there's a location for which I can get weather, implying a known city. Let's think if there's a creative way to use the get_weather tool.

Action:

```json
{
  "action": "get_weather",
  "action_input": {"location": "Paris"}
}
```

Observation: Assuming the get_weather tool returns the current weather in Paris, it indirectly confirms that Paris is indeed a location for which weather can be queried, implying it's a known city.

Thought: Knowing that Paris is the capital of France is a piece of general knowledge. The action I took was more of a confirmation that Paris is a significant city, but I didn't need the weather details to answer the question about the capital of France.

Thought: I now know the final answer
Final Answer: Paris


In [9]:
output = client.chat.completions.create(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "What's the weather in London?"},
    ],
    stream=False,
    max_tokens=1024,
    stop=["Observation:"]
)
print(output.choices[0].message.content)

Thought: To find out the weather in London, I should use the `get_weather` tool with the location set to "London".

Action:

```json
{
  "action": "get_weather",
  "action_input": {"location": "London"}
}
```




In [11]:
# Dummy function
def get_weather(location):
    return f"the weather in {location} is hot and cool.\n"

get_weather('London')

'the weather in London is hot and cool.\n'

In [ ]:
messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "What's the weather in London?"},
        {"role": "assistant", "content": output.choices[0].message.content + "Observation:\n" + get_weather("London")}

]
output = client.chat.completions.create(
    messages=messages,
    stream=False,
    max_tokens=1024,
    stop=["Observation:"]
)
print(output.choices[0].message.content)

Thought: I now know the final answer

Final Answer: the weather in London is hot and cool.
